# Option 3: Native Convolutional Neural Network (from scratch)

This notebook builds an image classifier using a plain, "native" convolutional neural network - no pretrained backbone, no transfer learning. The network is trained entirely from scratch on the dataset images. Because this is a multi-class classification problem, the output layer uses a `softmax` activation (one probability per class) and the model is compiled with `categorical_crossentropy` loss, so labels are one-hot encoded before training.

In [ ]:
import json
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split
from pathlib import Path

In [ ]:
image_size = (128, 128)
batch_size = 32
seed = 42

# Resolve the dataset path whether the notebook runs from the repo root or Assignment_2.
dataset_dir = Path("Assignment_2/ImageDataset/images")
if not dataset_dir.exists():
    dataset_dir = Path("ImageDataset/images")

image_dataset = tf.keras.utils.image_dataset_from_directory(
    str(dataset_dir),
    image_size=image_size,
    batch_size=batch_size,
    shuffle=True,
    seed=seed,
)

class_names = image_dataset.class_names
image_batches = []
label_batches = []

# Keep images and labels from the same batch so they stay correctly matched.
for images, labels in image_dataset:
    image_batches.append(images.numpy())
    label_batches.append(labels.numpy())

x_data = np.concatenate(image_batches, axis=0)
y_data = np.concatenate(label_batches, axis=0)

train_images, test_images, train_labels, test_labels = train_test_split(
    x_data,
    y_data,
    test_size=0.2,
    random_state=seed,
    stratify=y_data,
)

(x_train, y_train), (x_test, y_test) = (train_images, train_labels), (test_images, test_labels)

num_classes = len(class_names)

# No pretrained backbone here, so we just scale raw pixels to the 0..1 range.
image_train = x_train.astype("float32") / 255.0
image_test = x_test.astype("float32") / 255.0

# categorical_crossentropy expects one-hot encoded labels rather than integer class ids.
labels_train = to_categorical(y_train, num_classes=num_classes)
labels_test = to_categorical(y_test, num_classes=num_classes)

print(class_names)
print(image_train.shape, labels_train.shape)
print(image_test.shape, labels_test.shape)
print(image_train.dtype, image_train.min(), image_train.max())
print(image_test.dtype, image_test.min(), image_test.max())

In [ ]:
input_shape = image_train.shape[1:]

# Layer 1: Input layer - receives one RGB image scaled to 0..1.
inputs = Input(shape=input_shape, name="input_image")

# Block 1: Conv2D + ReLU learns low-level features (edges, colors, textures).
x = tf.keras.layers.Conv2D(32, (3, 3), activation="relu", padding="same", name="conv_block1")(inputs)
x = tf.keras.layers.MaxPooling2D((2, 2), name="pool_block1")(x)

# Block 2: More filters learn mid-level features (shapes, patterns).
x = tf.keras.layers.Conv2D(64, (3, 3), activation="relu", padding="same", name="conv_block2")(x)
x = tf.keras.layers.MaxPooling2D((2, 2), name="pool_block2")(x)

# Block 3: Even more filters learn higher-level, more abstract features.
x = tf.keras.layers.Conv2D(128, (3, 3), activation="relu", padding="same", name="conv_block3")(x)
x = tf.keras.layers.MaxPooling2D((2, 2), name="pool_block3")(x)

# Flatten the feature maps into a single feature vector.
x = tf.keras.layers.Flatten(name="flatten")(x)

# Dense layer - learns task-specific combinations of the CNN features.
x = tf.keras.layers.Dense(256, activation="relu", name="dense_features")(x)

# Dropout layer - helps reduce overfitting during training.
x = tf.keras.layers.Dropout(0.5, name="dropout_regularization")(x)

# Output layer - softmax produces one probability per class that sums to 1.
outputs = tf.keras.layers.Dense(num_classes, activation="softmax", name="class_probabilities")(x)

model = Model(inputs=inputs, outputs=outputs, name="animal_native_cnn")
model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

model.summary()

In [ ]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True,
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=2,
        min_lr=1e-6,
    ),
]

history = model.fit(
    image_train,
    labels_train,
    validation_data=(image_test, labels_test),
    epochs=30,
    batch_size=batch_size,
    callbacks=callbacks,
    verbose=1,
)

In [ ]:
test_loss, test_accuracy = model.evaluate(image_test, labels_test, verbose=0)
print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {test_accuracy:.4f}")

In [ ]:
model_dir = Path("Assignment_2")
if not model_dir.exists():
    model_dir = Path(".")

model_path = model_dir / "animal_native_cnn.keras"
weights_path = model_dir / "animal_native_cnn.weights.h5"
labels_path = model_dir / "class_names_native_cnn.json"

model.save(model_path)
model.save_weights(weights_path)

with open(labels_path, "w", encoding="utf-8") as labels_file:
    json.dump(class_names, labels_file, indent=2)

print(f"Saved full model to: {model_path.resolve()}")
print(f"Saved weights to: {weights_path.resolve()}")
print(f"Saved class labels to: {labels_path.resolve()}")

In [ ]:
import matplotlib.pyplot as plt

loaded_model = tf.keras.models.load_model(model_path)

with open(labels_path, "r", encoding="utf-8") as labels_file:
    saved_class_names = json.load(labels_file)

loaded_loss, loaded_accuracy = loaded_model.evaluate(image_test, labels_test, verbose=0)
print(f"Loaded model test loss: {loaded_loss:.4f}")
print(f"Loaded model test accuracy: {loaded_accuracy:.4f}")

sample_count = min(9, len(image_test))
sample_indices = np.arange(sample_count)
predictions = loaded_model.predict(image_test[sample_indices], verbose=0)
predicted_labels = np.argmax(predictions, axis=1)
true_labels = np.argmax(labels_test, axis=1)

plt.figure(figsize=(10, 10))
for plot_index, image_index in enumerate(sample_indices):
    true_label = saved_class_names[true_labels[image_index]]
    predicted_label = saved_class_names[predicted_labels[plot_index]]
    confidence = predictions[plot_index][predicted_labels[plot_index]]

    plt.subplot(3, 3, plot_index + 1)
    plt.imshow(x_test[image_index].astype("uint8"))
    plt.title(f"True: {true_label}\nPred: {predicted_label} ({confidence:.2f})")
    plt.axis("off")

plt.tight_layout()
plt.show()